# Short training pilots
CPU preparation first. Four fresh models, 2,000 updates each. Preserve every checkpoint and exact prediction on Drive. Inspect results and bot receipts before extending two survivors to 6,000; full training requires that comparison and a T4 inference check.

In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU first'
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess, sys
repo=Path('/content/hse-nlp-pilots')
subprocess.run(['git','clone','--depth=1','--branch=2026','https://github.com/OsVox/hse-nlp.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True)
subprocess.run([sys.executable,'-m','unittest','homeworks.hw2.tests.test_model_sizes','homeworks.hw2.tests.test_continuation'],check=True)
from homeworks.hw2.short_pilots import prepare
prepare()


Export and verify prepared/short-pilots-v1/ready.json from Drive. Only then select a BF16-capable training GPU with at least 40 GiB memory. Do not Run All.

In [ ]:
from google.colab import drive, runtime
try:
    drive.mount('/content/drive')
    import os, sys, json, hashlib, tarfile, subprocess
    from pathlib import Path
    prepared=Path('/content/drive/MyDrive/HSE_NLP_HW2/prepared/short-pilots-v1')
    ready=json.loads((prepared/'ready.json').read_text())
    assert ready['cpu_verified']
    payload=(prepared/'source.tar').read_bytes()
    assert hashlib.sha256(payload).hexdigest()==ready['source_sha256']
    work=Path('/content/hw2-short-pilots')
    work.mkdir(exist_ok=False)
    os.chdir(work)
    Path('source.tar').write_bytes(payload)
    with tarfile.open('source.tar') as archive: archive.extractall('.',filter='data')
    subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow>=18','requests>=2.31'],check=True,timeout=180)
    subprocess.run([sys.executable,'-u','-m','homeworks.hw2.short_pilots','--gpu'],check=True,timeout=3000)
finally:
    runtime.unassign()


After compute is released, export predictions and evidence from Drive, verify each manifest and exact token decode, submit each checkpoint once to the course bot, and save each receipt to Drive. Selection uses held-out training questions; bot feedback must not become training targets.